# AI-Driven Business Intelligence Framework for Advanced Decision Support
## Chicago Crime: Arrest-Status Prediction with Ensemble Learning

**Base paper:** *Crime Analyses Using Data Analytics* (Dayara, Thabtah, Abdel-Jaber, Zeidan, IJDWM 18(1)).
**Data:** Chicago Police Department "Crimes - 2001 to Present" export (`Crimes_-_2001_to_Present_20261013.csv`).

## 1. Project Introduction
Law-enforcement agencies record every reported incident, but only a minority end in an arrest. This notebook builds a Business Intelligence pipeline that (1) replicates the base paper's arrest-prediction workflow for battery and assault, (2) shows why its models were weak (severe class imbalance and few informative features), and (3) proposes ensemble models (Random Forest, Gradient Boosting, Stacking) that are compared against the replicated baselines. All outputs are exported as tidy CSVs for a Power BI dashboard.

## 2. Research Objective
1. Replicate the paper's pipeline: descriptive analysis, feature selection, Naive Bayes / C4.5 / RIPPER, SMOTE.
2. Quantify the class-imbalance problem the paper identified.
3. Test whether ensemble learning improves arrest prediction over the replicated baselines, on the **same split and same feature set** (a fair comparison), and also with an extended feature set.
4. Export predictions, comparisons and feature importance for Power BI.

## 3. Base Paper Methodology (what is replicated here)
| Paper step | This notebook |
|---|---|
| Remove rows with missing values; split `Date` into Month/Day/Time; drop ID, IUCR, Description, Ward, etc. | Section 6-7 |
| Descriptive analysis (crime type, hour, district/CA/beat) | Section 8 |
| Feature selection: LOOCV, Information Gain, CFS with Naive Bayes | Section 9 (mutual information = IG, plus a drop-one wrapper check) |
| Scope: battery and assault; 3 timeframes (00-06, 06-12, 12-24) | Section 7 |
| Naive Bayes, C4.5, RIPPER on Beat + Primary Type + Timeframe + Date | Section 10 |
| Imbalance (~80% "No arrest") makes accuracy misleading; apply SMOTE | Sections 11-12 |

**Deliberate deviations from the paper (all documented):**
* The paper applied SMOTE *before* cross-validation, which leaks synthetic copies of test rows into training. Here the data is split first and SMOTE is applied to the **training set only**; every metric comes from an untouched, still-imbalanced test set.
* C4.5 is approximated with scikit-learn's entropy-based `DecisionTreeClassifier` (binary splits on ordinal-coded categories).
* The paper used Chicago 2017-2019. The supplied file covers a different, shorter period (checked in Section 5), so absolute numbers are not directly comparable with the paper.

## 4. Dataset Loading

In [ ]:
!pip -q install imbalanced-learn wittgenstein

import os, warnings, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix)
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTEN, SMOTENC

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
N_EST = 300          # trees in Random Forest (lower to 100 for a faster run)
OUT_DIR = 'powerbi_data'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
DATA_PATH = 'Crimes_-_2001_to_Present_20261013.csv'   # upload to the Colab file panel, or Drive path
if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()                          # pick the Chicago CSV
    DATA_PATH = list(uploaded.keys())[0]

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print('Loaded:', df_raw.shape)
df_raw.head(3)

## 5. Data Understanding

In [ ]:
print(df_raw.dtypes, '\n')
print('Missing values per column (only columns with gaps):')
print(df_raw.isna().sum()[df_raw.isna().sum() > 0], '\n')
dts = pd.to_datetime(df_raw['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
print('Date range :', dts.min(), '->', dts.max())
print('Rows per year:'); print(df_raw['Year'].value_counts().sort_index(), '\n')
print('Arrest rate (all crimes):'); print(df_raw['Arrest'].astype(str).str.lower().value_counts(normalize=True).round(4))
print('\nPrimary Type values:', df_raw['Primary Type'].nunique())
df_raw['Primary Type'].value_counts().head(10)

## 6. Data Cleaning
Following the paper: drop rows with missing values in the fields we use, remove duplicates, and drop identifier / redundant columns (IUCR, Ward, FBI Code, Updated On, projected coordinates). `Domestic`, `Location Description` and `Description` are kept only for the *extended* feature set.

In [ ]:
df = df_raw.drop_duplicates(subset='ID').copy()
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
for c in ['Arrest', 'Domestic']:
    df[c] = df[c].astype(str).str.lower().map({'true': 1, 'false': 0})
df['Location Description'] = df['Location Description'].fillna('UNKNOWN')

required = ['Date', 'Arrest', 'Beat', 'District', 'Community Area', 'Primary Type']
before = len(df)
df = df.dropna(subset=required).copy()
print(f'Removed {before-len(df):,} rows with missing key fields ({(before-len(df))/before:.2%}); {len(df):,} remain')

for c in ['Beat', 'District', 'Community Area', 'Arrest', 'Domestic']:
    df[c] = df[c].astype(int)

df = df.drop(columns=['IUCR', 'Updated On', 'Ward', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Location', 'Year'], errors='ignore')
df.info()

## 7. Feature Engineering
As in the paper, `Date` is split into time parts and a 3-level **Timeframe** (00-06, 06-12, 12-24) is created. Additional features (Hour, weekend flag, grouped location type, grouped crime sub-type `Description`) are used only in the extended set.

In [ ]:
df['Hour'] = df['Date'].dt.hour
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.day_name()
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)
df['Timeframe'] = np.select([df['Hour'] < 6, df['Hour'] < 12],
                            ['00-06 Midnight-6am', '06-12 6am-Noon'], default='12-24 Noon-Midnight')
top_loc = df['Location Description'].value_counts().head(15).index
df['LocGroup'] = np.where(df['Location Description'].isin(top_loc), df['Location Description'], 'OTHER')
# crime sub-type (the paper dropped `Description`; it is known at report time, so it is used only in the Extended set)
top_desc = df['Description'].value_counts().head(25).index
df['DescGroup'] = np.where(df['Description'].isin(top_desc), df['Description'], 'OTHER')

TARGET_CRIMES = ['BATTERY', 'ASSAULT']
dm = df[df['Primary Type'].isin(TARGET_CRIMES)].copy().reset_index(drop=True)
print(f'All cleaned incidents: {len(df):,} | Battery + Assault modelling scope: {len(dm):,}')
dm[['Primary Type', 'Beat', 'District', 'Timeframe', 'Hour', 'Month', 'DayOfWeek', 'Arrest']].head()

## 8. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 10))
df['Primary Type'].value_counts().head(12).plot.barh(ax=ax[0, 0], color='#1f4e79'); ax[0, 0].invert_yaxis()
ax[0, 0].set_title('Incidents by crime type')
df.groupby('Hour').size().plot.bar(ax=ax[0, 1], color='#1f4e79'); ax[0, 1].set_title('Incidents by hour of day')
df.groupby('YearMonth').size().plot(ax=ax[1, 0], marker='o', color='#c0392b'); ax[1, 0].set_title('Monthly incident trend')
ax[1, 0].tick_params(axis='x', rotation=60)
(df.groupby('Primary Type')['Arrest'].mean().sort_values(ascending=False).head(12) * 100).plot.barh(ax=ax[1, 1], color='#2e8b57')
ax[1, 1].invert_yaxis(); ax[1, 1].set_title('Arrest rate (%) by crime type'); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
dm.groupby(['District', 'Primary Type']).size().unstack(fill_value=0).sort_values('BATTERY', ascending=False).head(10).plot.bar(stacked=True, ax=ax[0])
ax[0].set_title('Top districts: battery + assault')
pivot = dm.pivot_table(index='DayOfWeek', columns='Hour', values='Arrest', aggfunc='size').reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
sns.heatmap(pivot, cmap='YlOrRd', ax=ax[1]); ax[1].set_title('Battery + assault: day x hour')
(dm.groupby('Timeframe')['Arrest'].mean() * 100).plot.bar(ax=ax[2], color='#2e8b57')
ax[2].set_title('Arrest rate (%) by timeframe'); plt.tight_layout(); plt.show()

print('Top 10 beats by battery + assault volume (with arrest rate):')
dm.groupby('Beat').agg(Incidents=('Arrest', 'size'), ArrestRate=('Arrest', 'mean')).sort_values('Incidents', ascending=False).head(10).round(3)

## 9. Feature Selection
The paper used LOOCV, Information Gain and CFS with Naive Bayes. Here: **mutual information** (equivalent to information gain for discrete features) and a **drop-one-feature wrapper** (3-fold CV AUC of an entropy tree, on a 20k sample) replace LOOCV, which is impractical at this size. Note that mutual information is biased upward for high-cardinality features such as Beat.

In [ ]:
PAPER_FEATURES = ['Beat', 'Primary Type', 'Timeframe', 'Month', 'DayOfWeek']   # paper: Beat/Block, Primary Type, Timeframe, Date
EXT_EXTRA = ['District', 'Community Area', 'LocGroup', 'DescGroup', 'Domestic', 'IsWeekend']
EXTENDED_FEATURES = PAPER_FEATURES + EXT_EXTRA + ['Hour']
CAT_FEATURES = PAPER_FEATURES + EXT_EXTRA

enc = OrdinalEncoder()
Xall = dm[EXTENDED_FEATURES].copy()
Xall[CAT_FEATURES] = enc.fit_transform(dm[CAT_FEATURES].astype(str)).astype(int)
y = dm['Arrest'].astype(int)

mi = pd.Series(mutual_info_classif(Xall, y, discrete_features=True, random_state=RANDOM_STATE), index=EXTENDED_FEATURES).sort_values(ascending=False)

samp = Xall.sample(min(20000, len(Xall)), random_state=RANDOM_STATE); ys = y.loc[samp.index]
dt = DecisionTreeClassifier(criterion='entropy', min_samples_leaf=20, random_state=RANDOM_STATE)
full = cross_val_score(dt, samp, ys, cv=3, scoring='roc_auc').mean()
drop_impact = {f: full - cross_val_score(dt, samp.drop(columns=f), ys, cv=3, scoring='roc_auc').mean() for f in EXTENDED_FEATURES}
fs = pd.DataFrame({'MutualInfo': mi, 'AUC_drop_when_removed': pd.Series(drop_impact)}).sort_values('MutualInfo', ascending=False)
print(f'Full-feature CV AUC (tree, 20k sample): {full:.4f}')
fs.round(5)

In [ ]:
fs['MutualInfo'].sort_values().plot.barh(figsize=(8, 5), color='#1f4e79', title='Mutual information with Arrest')
plt.show()
print('Paper feature set   :', PAPER_FEATURES)
print('Extended feature set:', EXTENDED_FEATURES)

## 10. Base Paper Replication
One stratified 80/20 train/test split is shared by every model. First the three baselines are trained on the **raw, imbalanced** training data (the paper's Table 4a, which it called misleading).

* **Naive Bayes** - `CategoricalNB`
* **Decision Tree / C4.5 approximation** - entropy criterion, `min_samples_leaf=10`
* **RIPPER** - `wittgenstein` library on a 15k-row stratified subsample (rule induction is slow). If the library cannot be installed, a shallow tree is used as a clearly-labelled substitute.

In [ ]:
idx_tr, idx_te = train_test_split(np.arange(len(dm)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)
ytr, yte = y.iloc[idx_tr].reset_index(drop=True), y.iloc[idx_te].reset_index(drop=True)

SETS = {}
for name, feats in [('Paper', PAPER_FEATURES), ('Extended', EXTENDED_FEATURES)]:
    SETS[name] = dict(feats=feats,
                      Xtr=Xall.iloc[idx_tr][feats].reset_index(drop=True),
                      Xte=Xall.iloc[idx_te][feats].reset_index(drop=True),
                      min_cat=(Xall[feats].max().values + 1))
print('Train:', len(ytr), '| Test:', len(yte))

RESULTS, FITTED, CM = [], {}, {}

def evaluate(name, group, fset, training, model, Xte, yte):
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else None
    row = dict(Model=name, Group=group, FeatureSet=fset, Training=training,
               Accuracy=accuracy_score(yte, pred), BalancedAccuracy=balanced_accuracy_score(yte, pred),
               Precision_w=precision_score(yte, pred, average='weighted', zero_division=0),
               Recall_w=recall_score(yte, pred, average='weighted', zero_division=0),
               F1_w=f1_score(yte, pred, average='weighted', zero_division=0),
               Precision_Arrest=precision_score(yte, pred, zero_division=0),
               Recall_Arrest=recall_score(yte, pred, zero_division=0),
               F1_Arrest=f1_score(yte, pred, zero_division=0),
               ROC_AUC=roc_auc_score(yte, proba) if proba is not None else np.nan,
               PR_AUC=average_precision_score(yte, proba) if proba is not None else np.nan)
    RESULTS.append(row)
    CM[(name, fset, training)] = confusion_matrix(yte, pred, labels=[0, 1])
    FITTED[(name, fset, training)] = model
    return row

class RipperModel:
    """RIPPER via `wittgenstein`; falls back to a shallow tree if the library is unavailable or fails."""
    def __init__(self, max_rows=15000):
        self.max_rows, self.backend = max_rows, 'RIPPER (wittgenstein)'
    def fit(self, X, y):
        d = X.astype(str).copy(); d['Arrest'] = np.asarray(y)
        d = d.sample(min(self.max_rows, len(d)), random_state=RANDOM_STATE)
        try:
            import wittgenstein as lw
            self.m = lw.RIPPER(random_state=RANDOM_STATE); self.m.fit(d, class_feat='Arrest', pos_class=1)
        except Exception as e:
            print('RIPPER unavailable ->', type(e).__name__, '; using shallow-tree substitute')
            self.backend = 'Rule-tree substitute (depth 6)'
            self.m = DecisionTreeClassifier(max_depth=6, criterion='entropy', random_state=RANDOM_STATE).fit(X.loc[d.index], np.asarray(y)[[X.index.get_loc(i) for i in d.index]])
        return self
    def predict(self, X):
        if self.backend.startswith('RIPPER'):
            return np.array(self.m.predict(X.astype(str))).astype(int)
        return self.m.predict(X)

def baseline_models(s):
    return {'Naive Bayes': CategoricalNB(min_categories=s['min_cat']),
            'Decision Tree (C4.5 approx.)': DecisionTreeClassifier(criterion='entropy', min_samples_leaf=10, random_state=RANDOM_STATE),
            'RIPPER': RipperModel()}

s = SETS['Paper']
for name, m in baseline_models(s).items():
    m.fit(s['Xtr'], ytr)
    evaluate(name, 'Baseline (paper)', 'Paper', 'Imbalanced', m, s['Xte'], yte)

pd.DataFrame(RESULTS)[['Model', 'Training', 'Accuracy', 'Recall_Arrest', 'F1_Arrest', 'ROC_AUC']].round(4)

## 11. Class Imbalance Analysis

In [ ]:
cnt = ytr.value_counts().rename({0: 'No arrest', 1: 'Arrest'})
print('Training class balance:'); print(cnt.to_frame('rows').assign(share=(cnt / cnt.sum()).round(4)))
majority_acc = (yte == 0).mean()
print(f'\n"Always predict No-arrest" accuracy on the test set: {majority_acc:.4f}')
print('=> any model near this accuracy has learned almost nothing about arrests (the paper hit the same problem: ~80.22%).\n')

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
cnt.plot.bar(ax=ax[0], color=['#7f8c8d', '#c0392b'], title='Training class distribution')
for a, (k, v) in zip(ax[1:], [(('Naive Bayes', 'Paper', 'Imbalanced'), 'Naive Bayes'), (('Decision Tree (C4.5 approx.)', 'Paper', 'Imbalanced'), 'Decision Tree')]):
    sns.heatmap(CM[k], annot=True, fmt='d', cmap='Blues', ax=a, xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    a.set_title(f'{v} (imbalanced): confusion matrix'); a.set_xlabel('Predicted'); a.set_ylabel('Actual')
plt.tight_layout(); plt.show()

## 12. SMOTE
SMOTE is applied **to the training set only**. Because every feature is categorical or integer-coded, `SMOTEN` (all-categorical) is used for the paper feature set and `SMOTENC` (mixed) for the extended set, which has `Hour` as the one numeric column.

In [ ]:
def balance(X, y_, feats):
    cat_idx = [i for i, f in enumerate(feats) if f != 'Hour']
    sm = SMOTEN(random_state=RANDOM_STATE) if len(cat_idx) == len(feats) else SMOTENC(categorical_features=cat_idx, random_state=RANDOM_STATE)
    Xb, yb = sm.fit_resample(X, y_)
    Xb = pd.DataFrame(np.rint(np.asarray(Xb, dtype=float)).astype(int), columns=feats)
    return Xb, pd.Series(np.asarray(yb)).astype(int)

for k, s in SETS.items():
    s['Xbal'], s['ybal'] = balance(s['Xtr'], ytr, s['feats'])
    print(f"{k:9s} before: {ytr.value_counts().to_dict()}  ->  after SMOTE: {s['ybal'].value_counts().to_dict()}")

## 13. Baseline Model Evaluation (after SMOTE)
The same three baselines, trained on balanced data and scored on the untouched test set. This is the analogue of the paper's Table 4b.

In [ ]:
s = SETS['Paper']
for name, m in baseline_models(s).items():
    m.fit(s['Xbal'], s['ybal'])
    evaluate(name, 'Baseline (paper)', 'Paper', 'SMOTE', m, s['Xte'], yte)

res = pd.DataFrame(RESULTS)
cols = ['Model', 'Training', 'Accuracy', 'Precision_w', 'Recall_w', 'F1_w', 'Precision_Arrest', 'Recall_Arrest', 'F1_Arrest', 'ROC_AUC']
base_tbl = res[(res.Group == 'Baseline (paper)')][cols].round(4)
base_tbl

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, nm in zip(ax, ['Naive Bayes', 'Decision Tree (C4.5 approx.)', 'RIPPER']):
    sns.heatmap(CM[(nm, 'Paper', 'SMOTE')], annot=True, fmt='d', cmap='Blues', ax=a, xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    a.set_title(f'{nm} + SMOTE'); a.set_xlabel('Predicted'); a.set_ylabel('Actual')
plt.tight_layout(); plt.show()
print('Paper (Chicago 2017-2019, different data) reported accuracy after SMOTE: NB 59.78%, RIPPER 51.89%, C4.5 63.50%.')

## 14. Proposed Ensemble Models
* **Random Forest** - bagged trees
* **Gradient Boosting** - scikit-learn `HistGradientBoostingClassifier`
* **Stacking** - Random Forest + Gradient Boosting + Decision Tree, combined by a logistic-regression meta-learner (3-fold out-of-fold predictions)

Each ensemble is trained on **both** feature sets. The *Paper* set isolates the benefit of ensembling itself (same inputs as the baselines); the *Extended* set shows the added value of richer features.

In [ ]:
def ensemble_models():
    rf = lambda: RandomForestClassifier(n_estimators=N_EST, min_samples_leaf=3, max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE)
    gb = lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=31, l2_regularization=1.0,
                                                early_stopping=True, random_state=RANDOM_STATE)
    dtc = lambda: DecisionTreeClassifier(criterion='entropy', min_samples_leaf=10, random_state=RANDOM_STATE)
    return {'Random Forest': rf(), 'Gradient Boosting': gb(),
            'Stacking Ensemble': StackingClassifier(estimators=[('rf', rf()), ('gb', gb()), ('dt', dtc())],
                                                    final_estimator=LogisticRegression(max_iter=1000),
                                                    stack_method='predict_proba', cv=3, n_jobs=1)}

for fset in ['Paper', 'Extended']:
    s = SETS[fset]
    for name, m in ensemble_models().items():
        m.fit(s['Xbal'], s['ybal'])
        evaluate(name, 'Proposed ensemble', fset, 'SMOTE', m, s['Xte'], yte)
        print(f'done: {name} [{fset} features]')

## 15. Ensemble Evaluation

In [ ]:
res = pd.DataFrame(RESULTS)
ens_tbl = res[res.Group == 'Proposed ensemble'][['Model', 'FeatureSet'] + cols[2:]].round(4)
display(ens_tbl)

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
for a, (k, v) in zip(ax.ravel(), [(k, v) for k, v in CM.items() if k[2] == 'SMOTE' and k[0] in ('Random Forest', 'Gradient Boosting', 'Stacking Ensemble')]):
    sns.heatmap(v, annot=True, fmt='d', cmap='Greens', ax=a, xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    a.set_title(f'{k[0]} [{k[1]}]'); a.set_xlabel('Predicted'); a.set_ylabel('Actual')
plt.tight_layout(); plt.show()

## 16. Baseline vs Ensemble Comparison
Accuracy on this test set is dominated by the majority class, so judge models on **balanced accuracy, Arrest-class F1 and ROC-AUC** as well. Deltas are measured against the best SMOTE baseline on the *same* (Paper) feature set.

In [ ]:
res = pd.DataFrame(RESULTS)
cmp_ = res[res.Training == 'SMOTE'].copy()
cmp_['Label'] = cmp_['Model'] + ' [' + cmp_['FeatureSet'] + ']'
cmp_ = cmp_.sort_values('ROC_AUC', ascending=False, na_position='last')
display(cmp_[['Label', 'Group', 'Accuracy', 'BalancedAccuracy', 'F1_Arrest', 'Recall_Arrest', 'ROC_AUC', 'PR_AUC']].round(4).reset_index(drop=True))

best_base = res[(res.Group == 'Baseline (paper)') & (res.Training == 'SMOTE')].sort_values('BalancedAccuracy', ascending=False).iloc[0]
print(f"\nBest baseline (SMOTE): {best_base.Model}  acc={best_base.Accuracy:.4f}  bal.acc={best_base.BalancedAccuracy:.4f}  F1(arrest)={best_base.F1_Arrest:.4f}")
delta = res[res.Group == 'Proposed ensemble'].copy()
delta['dAccuracy'] = delta.Accuracy - best_base.Accuracy
delta['dBalancedAcc'] = delta.BalancedAccuracy - best_base.BalancedAccuracy
delta['dF1_Arrest'] = delta.F1_Arrest - best_base.F1_Arrest
display(delta[['Model', 'FeatureSet', 'dAccuracy', 'dBalancedAcc', 'dF1_Arrest']].round(4))

plot_df = cmp_.melt(id_vars='Label', value_vars=['Accuracy', 'BalancedAccuracy', 'F1_Arrest'], var_name='Metric')
plt.figure(figsize=(14, 5)); sns.barplot(data=plot_df, x='Label', y='value', hue='Metric')
plt.xticks(rotation=40, ha='right'); plt.title('Baselines vs ensembles (SMOTE-trained, imbalanced test set)'); plt.tight_layout(); plt.show()

## 17. Feature Importance / Explainability

In [ ]:
rf_ext = FITTED[('Random Forest', 'Extended', 'SMOTE')]
imp_rf = pd.Series(rf_ext.feature_importances_, index=SETS['Extended']['feats'])

ens_ok = res[(res.Group == 'Proposed ensemble')].sort_values('ROC_AUC', ascending=False).iloc[0]
BEST_KEY = (ens_ok.Model, ens_ok.FeatureSet, 'SMOTE')
best_model = FITTED[BEST_KEY]; bf = SETS[BEST_KEY[1]]['feats']
print('Best ensemble by ROC-AUC:', BEST_KEY)

sub = SETS[BEST_KEY[1]]['Xte'].sample(min(5000, len(yte)), random_state=RANDOM_STATE)
perm = permutation_importance(best_model, sub, yte.loc[sub.index], scoring='roc_auc', n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
imp_perm = pd.Series(perm.importances_mean, index=bf)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
imp_rf.sort_values().plot.barh(ax=ax[0], color='#1f4e79', title='Random Forest impurity importance (Extended)')
imp_perm.sort_values().plot.barh(ax=ax[1], color='#c0392b', title=f'Permutation importance: {BEST_KEY[0]} [{BEST_KEY[1]}]')
plt.tight_layout(); plt.show()

fi = pd.concat([imp_rf.rename('RF_Importance'), imp_perm.rename('Permutation_AUC_Drop')], axis=1).reset_index().rename(columns={'index': 'Feature'})
fi = fi.merge(fs.reset_index().rename(columns={'index': 'Feature'}), on='Feature', how='outer')
fi.round(5)

## 18. Generate Power BI Dataset
Three views are exported to `powerbi_data/`:
* `crime_cleaned.csv` - every cleaned incident (all crime types) with engineered features, for descriptive dashboards
* `crime_predictions.csv` - every battery/assault incident scored by the best ensemble, with a `Split` column (**only `Test` rows are out-of-sample**; filter on it for honest accuracy visuals)
* `model_comparison.csv`, `feature_importance.csv`, `confusion_matrices.csv` - for the model-performance pages

In [ ]:
pb_clean = df.drop(columns=['Date']).assign(Date=df['Date'].dt.strftime('%Y-%m-%d %H:%M:%S'))
pb_clean.to_csv(f'{OUT_DIR}/crime_cleaned.csv', index=False)

res.round(5).to_csv(f'{OUT_DIR}/model_comparison.csv', index=False)
fi.round(6).to_csv(f'{OUT_DIR}/feature_importance.csv', index=False)
cm_rows = [dict(Model=k[0], FeatureSet=k[1], Training=k[2], Actual=a, Predicted=p, Count=int(v[a, p]))
           for k, v in CM.items() for a in (0, 1) for p in (0, 1)]
pd.DataFrame(cm_rows).to_csv(f'{OUT_DIR}/confusion_matrices.csv', index=False)
print('Saved:', os.listdir(OUT_DIR))

## 19. Save Prediction Results

In [ ]:
Xscore = Xall[bf]
proba = best_model.predict_proba(Xscore)[:, 1]
pred = (proba >= 0.5).astype(int)
split = np.array(['Train'] * len(dm), dtype=object); split[idx_te] = 'Test'

pb_pred = dm[['ID', 'Case Number', 'Date', 'Block', 'Primary Type', 'DescGroup', 'Location Description', 'LocGroup', 'District', 'Beat',
              'Community Area', 'Latitude', 'Longitude', 'Hour', 'Month', 'DayOfWeek', 'Timeframe', 'Domestic']].copy()
pb_pred['Date'] = pb_pred['Date'].dt.strftime('%Y-%m-%d %H:%M:%S')
pb_pred['Actual_Arrest'] = dm['Arrest'].values
pb_pred['Predicted_Arrest'] = pred
pb_pred['Arrest_Probability'] = proba.round(4)
pb_pred['Correct'] = (pb_pred['Actual_Arrest'] == pb_pred['Predicted_Arrest']).astype(int)
pb_pred['Split'] = split
pb_pred['Model'] = f'{BEST_KEY[0]} [{BEST_KEY[1]}]'
pb_pred.to_csv(f'{OUT_DIR}/crime_predictions.csv', index=False)

import joblib; joblib.dump(dict(model=best_model, features=bf, encoder=enc, cat_features=CAT_FEATURES), 'best_arrest_model.joblib')

test_view = pb_pred[pb_pred.Split == 'Test']
print(f'Scored {len(pb_pred):,} incidents with {BEST_KEY[0]} [{BEST_KEY[1]}]; held-out test accuracy = {test_view.Correct.mean():.4f}')

with zipfile.ZipFile('powerbi_data.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT_DIR): z.write(os.path.join(OUT_DIR, f), f)
try:
    from google.colab import files; files.download('powerbi_data.zip')
except Exception:
    print('Download powerbi_data.zip from the Colab file panel.')

## 20. Final Findings

In [ ]:
res = pd.DataFrame(RESULTS)
nb_i = res[(res.Model == 'Naive Bayes') & (res.Training == 'Imbalanced')].iloc[0]
b = best_base
e_paper = res[(res.Group == 'Proposed ensemble') & (res.FeatureSet == 'Paper')].sort_values('ROC_AUC', ascending=False).iloc[0]
e_ext = res[(res.Group == 'Proposed ensemble') & (res.FeatureSet == 'Extended')].sort_values('ROC_AUC', ascending=False).iloc[0]
top3 = imp_perm.sort_values(ascending=False).head(3).index.tolist()
hot = dm.groupby('District').agg(n=('Arrest', 'size'), rate=('Arrest', 'mean')).sort_values('n', ascending=False).head(3)

print(f"""
FINAL FINDINGS
1. Scope: {len(dm):,} battery/assault incidents; overall arrest rate {y.mean():.1%}. Always predicting "No arrest" scores {majority_acc:.1%}.
2. Imbalance: baselines on raw data reach ~{nb_i.Accuracy:.1%} accuracy but find almost no arrests (Naive Bayes arrest-recall {nb_i.Recall_Arrest:.1%}); this reproduces the paper's "misleading accuracy" finding.
3. Best SMOTE baseline on paper features: {b.Model} - accuracy {b.Accuracy:.1%}, balanced accuracy {b.BalancedAccuracy:.1%}, arrest F1 {b.F1_Arrest:.3f}.
4. Ensembles, SAME paper features: best = {e_paper.Model} - accuracy {e_paper.Accuracy:.1%}, balanced accuracy {e_paper.BalancedAccuracy:.1%}, arrest F1 {e_paper.F1_Arrest:.3f}, ROC-AUC {e_paper.ROC_AUC:.3f}.
5. Ensembles, EXTENDED features: best = {e_ext.Model} - accuracy {e_ext.Accuracy:.1%}, balanced accuracy {e_ext.BalancedAccuracy:.1%}, arrest F1 {e_ext.F1_Arrest:.3f}, ROC-AUC {e_ext.ROC_AUC:.3f}.
6. Most influential features (permutation, {BEST_KEY[0]}): {', '.join(top3)}.
7. Highest-volume districts: {', '.join(f'D{int(d)} ({int(r.n):,} incidents, {r.rate:.1%} arrests)' for d, r in hot.iterrows())}.
""")

### Interpretation notes and limitations
* **Read improvements on the right metric.** SMOTE-trained models trade raw accuracy for arrest detection, so compare balanced accuracy, arrest-class F1/recall and ROC-AUC (Sections 15-16), not accuracy alone.
* **Features describe *when/where/what*, not *how the case unfolded*.** Arrests depend heavily on information unavailable at report time (suspect known, offender on scene), which caps achievable accuracy; the paper reached the same conclusion.
* **Short time window.** The file covers a limited period, so Month effects are confounded with the specific months present; do not read them as seasonality.
* **Beat has high cardinality**, which inflates mutual-information and impurity importance; permutation importance (Section 17) is the more reliable ranking.
* In the Power BI predictions file, only `Split = Test` rows are out-of-sample.